In [ ]:
!pip install -q openai-whisper librosa
# !apt-get install -y ffmpeg
!pip install ffmpeg-python


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
'apt-get' is not recognized as an internal or external command,
operable program or batch file.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import whisper
import librosa
import numpy as np
import re

# --------------------------------------------------
# LOAD WHISPER MODEL
# --------------------------------------------------

model = whisper.load_model("base")


# --------------------------------------------------
# FILLER WORDS
# --------------------------------------------------

FILLER_WORDS = [
    "um",
    "uh",
    "erm",
    "hmm",
    "like",
    "basically",
    "actually",
    "you know",
    "i mean",
    "sort of",
    "kind of"
]


# --------------------------------------------------
# 1. TRANSCRIBE AUDIO
# --------------------------------------------------

def transcribe_audio(audio_file):

    result = model.transcribe(
        audio_file,
        word_timestamps=True
    )

    return result


# --------------------------------------------------
# 2. FILLER WORD DETECTION
# --------------------------------------------------

def detect_fillers(text):

    text = text.lower()

    filler_counts = {}

    for filler in FILLER_WORDS:

        pattern = r"\b" + re.escape(filler) + r"\b"

        count = len(re.findall(pattern, text))

        if count > 0:
            filler_counts[filler] = count

    total_fillers = sum(filler_counts.values())

    return filler_counts, total_fillers


# --------------------------------------------------
# 3. REPEATED WORD / FUMBLE DETECTION
# --------------------------------------------------

def detect_repetitions(text):

    words = re.findall(r"\b[a-zA-Z]+\b", text.lower())

    repetitions = []

    for i in range(len(words) - 1):

        if words[i] == words[i + 1]:

            repetitions.append(words[i])

    return repetitions


# --------------------------------------------------
# 4. PAUSE DETECTION
# --------------------------------------------------

def detect_pauses(segments):

    pauses = []

    previous_end = None

    for segment in segments:

        start = segment["start"]
        end = segment["end"]

        if previous_end is not None:

            pause = start - previous_end

            if pause >= 1:

                pauses.append(pause)

        previous_end = end

    return pauses


# --------------------------------------------------
# 5. SPEAKING RATE
# --------------------------------------------------

def calculate_wpm(text, duration):

    words = re.findall(r"\b[a-zA-Z]+\b", text)

    word_count = len(words)

    minutes = duration / 60

    if minutes == 0:
        return 0

    wpm = word_count / minutes

    return round(wpm, 2)


# --------------------------------------------------
# 6. FLUENCY SCORE
# --------------------------------------------------

def calculate_fluency_score(
    total_words,
    filler_count,
    repetition_count,
    long_pause_count
):

    if total_words == 0:
        return 0

    penalty = (
        filler_count * 1.5
        + repetition_count * 2
        + long_pause_count * 3
    )

    score = 100 - penalty

    score = max(0, min(100, score))

    return round(score, 2)


# --------------------------------------------------
# 7. COMPLETE ANALYSIS
# --------------------------------------------------

def analyze_speech(audio_file):

    print("\nAnalyzing audio...\n")

    result = transcribe_audio(audio_file)

    text = result["text"]

    segments = result["segments"]

    # Duration
    audio, sr = librosa.load(
        audio_file,
        sr=None
    )

    duration = len(audio) / sr

    # Words
    words = re.findall(
        r"\b[a-zA-Z]+\b",
        text
    )

    total_words = len(words)

    # Fillers
    filler_counts, filler_count = detect_fillers(text)

    # Repetitions
    repetitions = detect_repetitions(text)

    repetition_count = len(repetitions)

    # Pauses
    pauses = detect_pauses(segments)

    long_pauses = [
        p for p in pauses
        if p >= 2
    ]

    average_pause = (
        np.mean(pauses)
        if pauses
        else 0
    )

    longest_pause = (
        max(pauses)
        if pauses
        else 0
    )

    # Speaking rate
    wpm = calculate_wpm(
        text,
        duration
    )

    # Fluency
    fluency_score = calculate_fluency_score(
        total_words,
        filler_count,
        repetition_count,
        len(long_pauses)
    )

    # --------------------------------------------------
    # PRINT RESULTS
    # --------------------------------------------------

    print("=" * 50)
    print("       SPEECH ANALYSIS REPORT")
    print("=" * 50)

    print(f"\nAudio Duration       : {duration:.2f} seconds")

    print(f"Total Words          : {total_words}")

    print(f"Speaking Rate        : {wpm} WPM")

    print("\n--- FILLER WORDS ---")

    print(f"Total Fillers        : {filler_count}")

    if filler_counts:

        for word, count in filler_counts.items():

            print(
                f"{word:<15} : {count}"
            )

    else:

        print("No filler words detected.")

    print("\n--- FUMBLE / REPETITION ---")

    print(
        f"Repeated Words      : {repetition_count}"
    )

    if repetitions:

        print(
            "Detected            :",
            ", ".join(repetitions)
        )

    else:

        print("No obvious repetitions.")

    print("\n--- PAUSE ANALYSIS ---")

    print(
        f"Total Pauses        : {len(pauses)}"
    )

    print(
        f"Long Pauses (>2s)   : {len(long_pauses)}"
    )

    print(
        f"Average Pause       : {average_pause:.2f} sec"
    )

    print(
        f"Longest Pause       : {longest_pause:.2f} sec"
    )

    print("\n--- FINAL SCORE ---")

    print(
        f"Fluency Score       : {fluency_score}/100"
    )

    print("=" * 50)

    return {

        "duration": round(duration, 2),

        "total_words": total_words,

        "wpm": wpm,

        "filler_count": filler_count,

        "filler_words": filler_counts,

        "repetition_count": repetition_count,

        "repetitions": repetitions,

        "total_pauses": len(pauses),

        "long_pauses": len(long_pauses),

        "average_pause": round(
            average_pause, 2
        ),

        "longest_pause": round(
            longest_pause, 2
        ),

        "fluency_score": fluency_score
    }


# --------------------------------------------------
# RUN PROGRAM
# --------------------------------------------------

audio_file = r"audio.wav"

result = analyze_speech(audio_file)


Analyzing audio...

       SPEECH ANALYSIS REPORT

Audio Duration       : 11.84 seconds
Total Words          : 24
Speaking Rate        : 121.62 WPM

--- FILLER WORDS ---
Total Fillers        : 0
No filler words detected.

--- FUMBLE / REPETITION ---
Repeated Words      : 1
Detected            : sql

--- PAUSE ANALYSIS ---
Total Pauses        : 0
Long Pauses (>2s)   : 0
Average Pause       : 0.00 sec
Longest Pause       : 0.00 sec

--- FINAL SCORE ---
Fluency Score       : 98.0/100


In [8]:
import os
os.getcwd()

'C:\\Users\\ADMIN\\OneDrive\\Desktop\\pga 12 python'

In [9]:
import os

audio_file = r"C:\Users\ADMIN\OneDrive\Desktop\pga 12 python\audio.wav"

print("File exists:", os.path.exists(audio_file))

File exists: True


In [10]:
!ffmpeg -i "C:\Users\ADMIN\OneDrive\Desktop\pga 12 python\audio.wav"

ffmpeg version 9.0.1-full_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers
  built with gcc 16.1.0 (Rev2, Built by MSYS2 project)
  configuration: --enable-gpl --enable-version3 --enable-static --disable-w32threads --disable-autodetect --enable-cairo --enable-fontconfig --enable-iconv --enable-gnutls --enable-lcms2 --enable-libxml2 --enable-gmp --enable-bzlib --enable-lzma --enable-libsnappy --enable-zlib --enable-librist --enable-libsrt --enable-libssh --enable-libzmq --enable-avisynth --enable-libbluray --enable-libcaca --enable-libdvdnav --enable-libdvdread --enable-sdl2 --enable-libaribb24 --enable-libaribcaption --enable-libdav1d --enable-libdavs2 --enable-libopenjpeg --enable-libquirc --enable-libuavs3d --enable-libxevd --enable-libzvbi --enable-liboapv --enable-libqrencode --enable-librav1e --enable-libsvtav1 --enable-libvvenc --enable-libwebp --enable-libx264 --enable-libx265 --enable-libxavs2 --enable-libxeve --enable-libxvid --enable-libaom --enable-libjxl --e

In [11]:
import whisper

model = whisper.load_model("base")

audio_file = r"C:\Users\ADMIN\OneDrive\Desktop\pga 12 python\audio.wav"

result = model.transcribe(
    audio_file,
    word_timestamps=True
)

print(result["text"])

 What is SQL? SQL stands for structured query language. It is a language used to store, retrieve, update and manage data in relational determinuses.
